# Unit 2 — Advanced Visualization & Storytelling
## Interactive Retail Sales Dashboard


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import ipywidgets as widgets
from ipywidgets import interact
sns.set_theme(style='whitegrid')
np.random.seed(7)
stores=pd.DataFrame({'store_id':['S01','S02','S03','S04','S05','S06'],'store_city':['Bengaluru','Mysuru','Mumbai','Pune','Delhi','Jaipur'],'region':['South','South','West','West','North','North']})
products=pd.DataFrame({'product':['Wireless Earbuds','Smartwatch','Laptop Sleeve','USB-C Hub','Yoga Mat','Dumbbell Set','Running Shoes','Track Jacket','Air Fryer','Blender'],'category':['Electronics']*4+['Fitness']*2+['Apparel']*2+['Home']*2,'unit_price':[1499,4999,899,1299,799,2499,3499,1999,3999,2299]})
months=pd.date_range('2023-01-01',periods=24,freq='MS'); rows=[]
for _,s in stores.iterrows():
    for _,p in products.iterrows():
        for i,m in enumerate(months):
            units=float(np.random.poisson(15*(1+0.25*np.sin(2*np.pi*m.month/12)))); revenue=units*p['unit_price']; profit=revenue*np.random.uniform(.12,.35); rating=np.clip(np.random.normal(4.1,.4),1,5)
            rows.append((m,s.store_id,s.store_city,s.region,p['product'],p['category'],p.unit_price,units,revenue,profit,round(rating,1)))
sales_df=pd.DataFrame(rows,columns=['month','store_id','store_city','region','product','category','unit_price','units_sold','revenue','profit','avg_rating'])
mask=(sales_df.store_id=='S03')&(sales_df.product=='Smartwatch')&(sales_df.month=='2024-06-01'); sales_df.loc[mask,['units_sold','revenue','profit']]*=.15


In [ ]:
revenue_by_product=sales_df.groupby('product').revenue.sum().sort_values(ascending=False)
sns.barplot(x=revenue_by_product.values,y=revenue_by_product.index,color='steelblue'); plt.title('Total Revenue by Product'); plt.tight_layout(); plt.show()
rev_cat=sales_df.groupby(['product','category'],as_index=False).revenue.sum().sort_values('revenue',ascending=False)
px.bar(rev_cat,x='product',y='revenue',color='category',hover_data={'revenue':':,.0f'}).show()


In [ ]:
price_slider=widgets.FloatRangeSlider(value=[sales_df.unit_price.min(),sales_df.unit_price.max()],min=sales_df.unit_price.min(),max=sales_df.unit_price.max(),step=50,description='Price:')
units_slider=widgets.FloatRangeSlider(value=[sales_df.units_sold.min(),sales_df.units_sold.max()],min=sales_df.units_sold.min(),max=sales_df.units_sold.max(),step=1,description='Units:')
def dynamic_filter(price_range,units_range):
    filtered=sales_df[sales_df.unit_price.between(*price_range)&sales_df.units_sold.between(*units_range)]; print(f'{len(filtered)} / {len(sales_df)} rows match')
    plt.scatter(sales_df.unit_price,sales_df.units_sold,c='lightgray',s=10); plt.scatter(filtered.unit_price,filtered.units_sold,c='crimson',s=25); plt.xlabel('Unit Price'); plt.ylabel('Units Sold'); plt.show()
interact(dynamic_filter,price_range=price_slider,units_range=units_slider)


In [ ]:
store_overview=sales_df.groupby('store_id').revenue.sum().reset_index()
def store_coordinated_view(selected_store):
    fig,axes=plt.subplots(1,3,figsize=(16,4)); axes[0].bar(store_overview.store_id,store_overview.revenue,color=['crimson' if s==selected_store else 'lightgray' for s in store_overview.store_id]); axes[0].set_title('Revenue by Store')
    cat=sales_df[sales_df.store_id==selected_store].groupby('category').revenue.sum(); axes[1].pie(cat,labels=cat.index,autopct='%1.0f%%'); axes[1].set_title('Category Breakdown')
    trend=sales_df[sales_df.store_id==selected_store].groupby('month').revenue.sum(); axes[2].plot(trend.index,trend.values,marker='o',color='crimson'); axes[2].tick_params(axis='x',rotation=45); axes[2].set_title('Monthly Trend'); plt.tight_layout(); plt.show()
interact(store_coordinated_view,selected_store=widgets.Dropdown(options=list(stores.store_id),description='Store:'))


In [ ]:
def min_median_ratio(s):
    med=s.median(); return s.min()/med if med else np.nan
anomaly_ranking=sales_df.groupby(['store_id','product']).units_sold.agg(min_median_ratio).sort_values().head(5); print(anomaly_ranking)
top_store,top_product=anomaly_ranking.index[0]; focus=sales_df[(sales_df.store_id==top_store)&(sales_df.product==top_product)].sort_values('month'); context=sales_df[sales_df.product==top_product].groupby('month').units_sold.mean()
plt.plot(context.index,context.values,color='gray',label='Context'); plt.plot(focus.month,focus.units_sold,color='crimson',marker='o',label='Focus'); plt.legend(); plt.title(f'{top_store} / {top_product}'); plt.show()


In [ ]:
product_summary=sales_df.groupby('product',as_index=False).agg(revenue=('revenue','sum'),profit=('profit','sum'),avg_rating=('avg_rating','mean')).sort_values('revenue',ascending=False); product_summary.style.bar(subset=['revenue'],color='#5DADE2').bar(subset=['profit'],color='#58D68D').bar(subset=['avg_rating'],color='#F5B041').set_caption('Product performance sorted by revenue')
product_summary.sort_values('profit',ascending=False).style.bar(subset=['revenue'],color='#5DADE2').bar(subset=['profit'],color='#58D68D').bar(subset=['avg_rating'],color='#F5B041')


In [ ]:
fig,axes=plt.subplots(2,2,figsize=(12,8)); axes[0,0].axis('off'); axes[0,0].text(.05,.5,f'Total revenue: ${sales_df.revenue.sum():,.0f}\nTotal profit: ${sales_df.profit.sum():,.0f}\nAvg rating: {sales_df.avg_rating.mean():.2f}\nActive stores: {sales_df.store_id.nunique()}',fontsize=13); axes[0,0].set_title('KPI Summary')
region_rev=sales_df.groupby('region').revenue.sum().sort_values(ascending=False); axes[0,1].bar(region_rev.index,region_rev.values,color='teal'); axes[0,1].set_title('Revenue by Region')
month_rev=sales_df.groupby('month').revenue.sum(); axes[1,0].plot(month_rev.index,month_rev.values,color='navy'); axes[1,0].set_title('Revenue by Month'); axes[1,0].tick_params(axis='x',rotation=45)
ratios=sales_df.groupby(['store_id','product']).units_sold.agg(min_median_ratio); n_alerts=int((ratios<.25).sum()); axes[1,1].axis('off'); axes[1,1].add_patch(mpatches.Circle((.5,.5),.35,color='red' if n_alerts else 'green')); axes[1,1].text(.5,.5,f'ALERT — {n_alerts} issue(s) found' if n_alerts else 'OK',ha='center',va='center',color='white',weight='bold'); plt.tight_layout(); plt.show()
